In [9]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import classification_report, confusion_matrix
import warnings

warnings.filterwarnings("ignore")

# ====================== 1. LOAD + CLEAN DATA ======================
print("Loading dataset...")
df = pd.read_excel("C:/Users/khaled/Downloads/gp/wahd.xlsx", header=1)

df.columns = df.columns.astype(str).str.strip().str.lower()

df = df.rename(columns={
    "boolean fire": "fire",
    "boolean ambulance": "ambulance",
    "boolean police": "police"
})

df = df.dropna(subset=["description", "severity"]).copy()

df["description"] = df["description"].astype(str).str.strip()
df["severity"] = df["severity"].astype(str).str.lower().str.strip()

_tf = {"true": 1, "false": 0, "TRUE": 1, "FALSE": 0, True: 1, False: 0}
for col in ["fire", "ambulance", "police"]:
    df[col] = df[col].map(_tf).fillna(0).astype(int)

allowed_sev = ["low", "urgent", "critical", "fake"]
df = df[df["severity"].isin(allowed_sev)].reset_index(drop=True)

# Professor rule: fake => no dispatch
df.loc[df["severity"] == "fake", ["fire", "ambulance", "police"]] = 0

# ====================== 2. TRAIN / TEST SPLIT ======================
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["severity"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

# ====================== 3. SEVERITY OVERSAMPLING (CLEAN) ======================
print("Balancing severity classes...")

sev_target = train_df["severity"].value_counts().max()

sev_train_df = pd.concat(
    [
        resample(group,
                 replace=True,
                 n_samples=sev_target,
                 random_state=42)
        for _, group in train_df.groupby("severity")
    ],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

# For dispatch we use the same balanced dataset
disp_train_df = sev_train_df.copy()

# ====================== 4. ENCODE TEXT ======================
print("Encoding text...")

embedder = SentenceTransformer("all-mpnet-base-v2")

X_sev_train = embedder.encode(sev_train_df["description"].tolist(), show_progress_bar=True)
X_disp_train = embedder.encode(disp_train_df["description"].tolist(), show_progress_bar=True)

X_sev_test = embedder.encode(test_df["description"].tolist(), show_progress_bar=True)
X_disp_test = embedder.encode(test_df["description"].tolist(), show_progress_bar=True)

y_sev_train = sev_train_df["severity"].values
y_disp_train = disp_train_df[["fire", "ambulance", "police"]].values

# ====================== 5. TRAIN MODELS ======================
print("Training models...")

# Severity classifier
sev_clf = LogisticRegression(
    max_iter=5000,
    C=0.15,
    class_weight="balanced",
    random_state=42
)
sev_clf.fit(X_sev_train, y_sev_train)

# Dispatch classifier (multi-label)
dispatch_clf = MultiOutputClassifier(
    LogisticRegression(
        max_iter=5000,
        C=0.15,
        class_weight="balanced",
        random_state=42
    ),
    n_jobs=1
)
dispatch_clf.fit(X_disp_train, y_disp_train)

# ====================== 6. EVALUATION ======================
print("\n" + "="*80)
print("EVALUATION ON REAL TEST SET")
print("="*80)

# ---- Severity ----
y_sev_true = test_df["severity"].values
y_sev_pred = sev_clf.predict(X_sev_test)

print("\n=== SEVERITY REPORT ===")
print(classification_report(y_sev_true, y_sev_pred, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_sev_true, y_sev_pred))

# ---- Dispatch ----
y_disp_true = test_df[["fire", "ambulance", "police"]].values
y_disp_pred = dispatch_clf.predict(X_disp_test)

labels = ["FIRE", "AMBULANCE", "POLICE"]

for i, label in enumerate(labels):
    print(f"\n=== {label} REPORT ===")
    print(classification_report(
        y_disp_true[:, i],
        y_disp_pred[:, i],
        digits=4
    ))

# ====================== 7. PREDICTION FUNCTION ======================
def predict_incident(text: str):
    emb = embedder.encode([text])

    sev = sev_clf.predict(emb)[0]
    disp = dispatch_clf.predict(emb)[0]

    if sev.upper() == "FAKE":
        return f"Input: {text}\n→ Severity: FAKE ALARM\n→ System Action: DO NOT RESPOND."

    units = []
    if disp[0]: units.append("FIRE")
    if disp[1]: units.append("AMBULANCE")
    if disp[2]: units.append("POLICE")

    if len(units) == 0:
        units = ["NONE"]

    return f"Input: {text}\n→ Severity: {sev.upper()}\n→ Units Dispatched: {units}"

# ====================== 8. SAMPLE TESTS ======================
print("\n" + "="*80)
print("SAMPLE PREDICTIONS")
print("="*80)

samples = [
    "I smell something burning behind the wall.",
    "There is an active shooter in the mall.",
    "My child has a mild fever.",
    "This is a prank call.",
     "A man just fell off scaffolding and is bleeding from his head, he is not waking up.",
    "Two cars collided head-on at high speed, people are trapped inside.",
    "There is an active shooter in the shopping mall, please hurry!",
    "The entire roof of the warehouse is engulfed in flames and collapsing.",
    "My kitchen stove is on fire, but it hasn't spread to the walls yet.",
    "Someone is trying to break into my neighbor's house with a crowbar.",
    "I sliced my hand open with a chef's knife and it's bleeding heavily.",
    "A guy just snatched my purse and ran down the alleyway.",
    "I've had a persistent cough for three weeks and my throat is scratchy.",
    "I stepped on a rusty nail but I cleaned it, just want to get it checked out.",
    "My child has a slight fever and a runny nose, but is otherwise playing.",
    "I tweaked my lower back lifting a heavy box, I can walk but it hurts.",
    "My husband was choking but he coughed it out and is breathing perfectly fine now.",
    "We had a small grease fire but we put it out with an extinguisher, nothing is burning.",
    "He passed out from the heat but drank some water and feels completely normal now.",
    "I locked my keys inside my car and I need someone to open it.",
    "The neighbors are shooting a music video with fake prop guns.",
    "My internet went down and I am going to miss an important work meeting.",
    "I am NOT having a heart attack, I just ate a really spicy burrito.",
    "Can you tell me what time the local pharmacy closes?",
     "There is smoke in the building but no fire visible yet",
    "He had a gun but nobody is injured",
    "She collapsed but says she feels better now",
    "The alarms are going off but nothing is burning",
    "My phone is dead",

    # FAKE
    "This is only a fire drill",
    "We are filming a scene for a movie",
    "This is a training simulation",
    "They are pretending to be unconscious",
    "This is a prank call",
    "I smell something burning behind the wall.",
    "There is a strong odor of melted plastic in the room.",
    "The electrical outlet smells hot and smoky.",
    "The air smells like something is overheating.",
    "There is a faint smoke-like odor in the hallway."
]

for s in samples:
    print(predict_incident(s))
    print("-"*60)

Loading dataset...
Train size: 3398
Test size: 850
Balancing severity classes...
Encoding text...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/144 [00:00<?, ?it/s]

Batches:   0%|          | 0/144 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Training models...

EVALUATION ON REAL TEST SET

=== SEVERITY REPORT ===
              precision    recall  f1-score   support

    critical     0.9175    0.8832    0.9000       214
        fake     0.8608    0.9007    0.8803       151
         low     0.8731    0.8643    0.8687       199
      urgent     0.8478    0.8566    0.8522       286

    accuracy                         0.8729       850
   macro avg     0.8748    0.8762    0.8753       850
weighted avg     0.8735    0.8729    0.8731       850

Confusion Matrix:
[[189   5   3  17]
 [  3 136   4   8]
 [  0   8 172  19]
 [ 14   9  18 245]]

=== FIRE REPORT ===
              precision    recall  f1-score   support

           0     0.9930    0.9627    0.9776       590
           1     0.9209    0.9846    0.9517       260

    accuracy                         0.9694       850
   macro avg     0.9569    0.9737    0.9646       850
weighted avg     0.9709    0.9694    0.9697       850


=== AMBULANCE REPORT ===
              precision